# Day 4: Fund Performance Analytics & Risk Metrics
This notebook loads historical NAV and Benchmark records from db/bluestock_mf.db, derives risk-return analytics (CAGR, Sharpe, Sortino, Alpha, Beta, Max Drawdown), validates against act_performance, constructs a 0-100 composite Fund Scorecard, and visualizes benchmark tracking errors.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

DB_PATH = Path('../db/bluestock_mf.db')
conn = sqlite3.connect(DB_PATH)

nav_df = pd.read_sql_query('SELECT * FROM fact_nav ORDER BY amfi_code, date', conn)
bench_df = pd.read_sql_query('SELECT * FROM fact_benchmark ORDER BY index_name, date', conn)
perf_df = pd.read_sql_query('SELECT * FROM fact_performance', conn)
fund_df = pd.read_sql_query('SELECT * FROM dim_fund', conn)
conn.close()

print(f'NAV records: {len(nav_df)}, Benchmark records: {len(bench_df)}, Funds: {len(perf_df)}')


## 1. Load and Inspect Fund Scorecard
Displaying the top 10 ranked funds based on the composite weighting formula: 30% 3Y CAGR + 25% Sharpe + 20% Alpha + 15% Inverted TER + 10% Inverted Max Drawdown.

In [ ]:
scorecard_df = pd.read_csv('../data/processed/fund_scorecard.csv')
scorecard_df.head(10)


## 2. Benchmark Comparison & Tracking Error
Plotting top 5 funds by 3-year return against NIFTY 50 and NIFTY 100.

In [ ]:
# Display tracking errors for top 5 funds
top5 = scorecard_df.sort_values('return_3yr_pct', ascending=False).head(5)
top5[['amfi_code', 'scheme_name', 'category', 'return_3yr_pct', 'composite_score']]
